# Benchmark Analysis: Subgraph Strategies for Link Prediction

This notebook reads pre-computed results from `benchmark_runner.ipynb` (and
optionally from an external `RESULT_DIR`) and produces a unified comparison.

**No training happens here** — only data loading and visualization.

## 0. Setup & Data Loading

**Best-config rule**: For methods with multiple configurations (Static PPR
has 5 `top_k` values, k-hop has 3 `k` values), the **summary figures** show
only the **best config per (dataset, encoder)** selected by highest test MRR.
The sensitivity figures (Sections 4–5) show all configs.

In [ ]:
import json, os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from collections import defaultdict

# ── Style ──
sns.set_theme(style='whitegrid', font_scale=1.05)
ENCODER_COLORS = {'GCN': '#1b9e77', 'SAGE': '#d95f02', 'GAT': '#7570b3'}
METHOD_MARKERS = {
    'Full Graph': 'o', 'Static PPR': 's',
    'Static k-hop': '^', 'Learnable PPR': 'D',
}
FIGW = 7

# ── Directories to scan for full_results.json ──
# Each maps a directory prefix to a (method_name, classify_fn).
# classify_fn(r) -> (config_label, config_value) from the loaded JSON dict.
RESULT_DIRS = {
    'results/benchmark':               'Full Graph',
    'results/benchmark-ppr':           'Static PPR',
    'results/benchmark-khop':          'Static k-hop',
    'results/benchmark-learnable-ppr': 'Learnable PPR',
}

# Optional: extra directory with full_results.json from other runs.
EXTRA_RESULT_DIR = '<result_dir>'  # <-- FILL THIS IN (or set to None)

def _classify(r, method):
    """Return (config_label, config_value) from a full_results.json dict."""
    if method == 'Static PPR':
        tk = r.get('top_k', r.get('config', {}).get('top_k', '?'))
        return f'top_k={tk}', tk
    if method == 'Static k-hop':
        k = r.get('k', r.get('config', {}).get('k', '?'))
        return f'k={k}', k
    if method == 'Learnable PPR':
        tk = r.get('top_k', '?')
        return f'top_k={tk}', tk
    return '-', None

def _detect_method(r):
    """Guess method from a JSON dict when loaded from EXTRA_RESULT_DIR."""
    if 'teleport_values' in r:
        return 'Learnable PPR'
    if 'k' in r or r.get('config', {}).get('k'):
        return 'Static k-hop'
    if 'top_k' in r or 'ppr_alpha' in r.get('config', {}):
        return 'Static PPR'
    return 'Full Graph'

def _parse_result(r, method):
    """Turn a full_results.json dict into a flat row."""
    tr = r.get('test_results', {})
    cfg, cfg_val = _classify(r, method)
    return {
        'method': method,
        'dataset': r.get('dataset', '?'),
        'encoder': r.get('encoder', '?'),
        'config': cfg,
        'config_value': cfg_val,
        'mrr':      float(tr.get('mrr', 0)),
        'auc':      float(tr.get('auc', 0)),
        'ap':       float(tr.get('ap', 0)),
        'hits@1':   float(tr.get('hits@1', 0)),
        'hits@3':   float(tr.get('hits@3', 0)),
        'hits@10':  float(tr.get('hits@10', 0)),
        'hits@50':  float(tr.get('hits@50', 0)),
        'hits@100': float(tr.get('hits@100', 0)),
        'train_time': float(r.get('train_time', r.get('search_time', 0))),
        'seed': r.get('seed', '?'),
        'run_id': r.get('run_id', ''),
        'timestamp': r.get('timestamp', ''),
    }

# ── Scan standard directories ──
# Each experiment dir has: full_results.json (latest) + runs/{timestamp}/full_results.json
# We only read the top-level full_results.json to avoid duplicates.
raw = []
for base_dir, method in RESULT_DIRS.items():
    if not os.path.isdir(base_dir):
        print(f'  {base_dir}: not found, skipping')
        continue
    count = 0
    for root, dirs, files in os.walk(base_dir):
        if 'runs' in dirs:
            dirs.remove('runs')  # don't descend into timestamped copies
        if 'full_results.json' in files:
            fpath = os.path.join(root, 'full_results.json')
            try:
                with open(fpath) as f:
                    r = json.load(f)
                raw.append(_parse_result(r, method))
                count += 1
            except Exception as e:
                print(f'  Error: {fpath}: {e}')
    print(f'  {base_dir}: {count} experiments')

# ── Scan extra directory ──
if EXTRA_RESULT_DIR and EXTRA_RESULT_DIR != '<result_dir>' and os.path.isdir(EXTRA_RESULT_DIR):
    count = 0
    for root, _, files in os.walk(EXTRA_RESULT_DIR):
        if 'full_results.json' in files:
            fpath = os.path.join(root, 'full_results.json')
            try:
                with open(fpath) as f:
                    r = json.load(f)
                method = _detect_method(r)
                raw.append(_parse_result(r, method))
                count += 1
            except Exception as e:
                print(f'  Error: {fpath}: {e}')
    print(f'  {EXTRA_RESULT_DIR}: {count} extra experiments')

df = pd.DataFrame(raw)
print(f'\nTotal rows: {len(df)}')
if not df.empty:
    print(f'Datasets:  {sorted(df["dataset"].unique())}')
    print(f'Encoders:  {sorted(df["encoder"].unique())}')
    print(f'Methods:   {sorted(df["method"].unique())}')

In [ ]:
# For each (method, dataset, encoder), keep only the row with highest MRR
# This collapses PPR top_k and k-hop k to the single best.

if not df.empty:
    idx_best = df.groupby(['method', 'dataset', 'encoder'])['mrr'].idxmax()
    df_best = df.loc[idx_best].reset_index(drop=True)

    print('Best config mapping (used in summary figures):')
    for _, row in df_best.iterrows():
        if row['config'] != '-':
            print(f'  {row["dataset"]:10s} | {row["encoder"]:5s} | '
                  f'{row["method"]:16s} -> {row["config"]}')
else:
    df_best = df

## 1. Summary Leaderboard (All Metrics)

In [ ]:
if not df_best.empty:
    metric_cols = ['mrr', 'auc', 'ap', 'hits@1', 'hits@10', 'hits@50']
    avail_metrics = [c for c in metric_cols if c in df_best.columns]
    display_cols = ['dataset', 'encoder', 'method', 'config'] + avail_metrics + ['train_time']
    display_cols = [c for c in display_cols if c in df_best.columns]

    tbl = df_best[display_cols].sort_values(['dataset', 'encoder', 'method'])

    fmt = {m: '{:.4f}' for m in avail_metrics}
    fmt['train_time'] = '{:.0f}s'

    styled = (tbl.style
              .format(fmt)
              .highlight_max(subset=['mrr'], color='#b6d7a8', axis=0))
    display(styled)
else:
    print('No results loaded.')

## 2. Headline: Best MRR by Method

Each bar is the **best config** for that method (e.g., PPR with optimal `top_k`).
Faceted by dataset.

In [ ]:
if not df_best.empty:
    ds_list = sorted(df_best['dataset'].unique())
    n_ds = len(ds_list)
    fig, axes = plt.subplots(1, n_ds, figsize=(FIGW * n_ds, 5), squeeze=False)

    for i, ds in enumerate(ds_list):
        ax = axes[0, i]
        sub = df_best[df_best['dataset'] == ds].sort_values('mrr', ascending=True)
        labels = sub['method'] + '\n' + sub['encoder']
        colors = [ENCODER_COLORS.get(e, '#999') for e in sub['encoder']]
        ax.barh(range(len(sub)), sub['mrr'], color=colors)
        ax.set_yticks(range(len(sub)))
        ax.set_yticklabels(labels, fontsize=9)
        ax.set_xlabel('MRR')
        ax.set_title(ds, fontsize=13, fontweight='bold')
        ax.set_xlim(left=max(0, sub['mrr'].min() - 0.02))
        for j, v in enumerate(sub['mrr']):
            ax.text(v + 0.002, j, f'{v:.4f}', va='center', fontsize=8)

    fig.suptitle('Test MRR — Best Config per Method',
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

## 3. Encoder Sensitivity

Same data as Section 2 but pivoted: **faceted by encoder** so you can see
whether conclusions are encoder-specific.

In [ ]:
if not df_best.empty:
    enc_list = sorted(df_best['encoder'].unique())
    n_enc = len(enc_list)
    fig, axes = plt.subplots(1, n_enc, figsize=(FIGW * n_enc, 4.5), squeeze=False)

    for i, enc in enumerate(enc_list):
        ax = axes[0, i]
        sub = df_best[df_best['encoder'] == enc].copy()
        sub['label'] = sub['dataset'] + ' / ' + sub['method']
        sub = sub.sort_values('mrr', ascending=True)
        ax.barh(range(len(sub)), sub['mrr'],
                color=ENCODER_COLORS.get(enc, '#999'), alpha=0.85)
        ax.set_yticks(range(len(sub)))
        ax.set_yticklabels(sub['label'], fontsize=9)
        ax.set_xlabel('MRR')
        ax.set_title(enc, fontsize=13, fontweight='bold')
        ax.set_xlim(left=max(0, sub['mrr'].min() - 0.02))

    fig.suptitle('MRR by Dataset & Method — per Encoder',
                 fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

## 4. Static PPR: MRR vs `top_k`

How sensitive is PPR performance to the subgraph size? One line per encoder,
faceted by dataset.

In [ ]:
ppr_df = df[df['method'] == 'Static PPR'].copy()
if not ppr_df.empty and 'config_value' in ppr_df.columns:
    ppr_df['top_k'] = pd.to_numeric(ppr_df['config_value'], errors='coerce')
    ppr_df = ppr_df.dropna(subset=['top_k'])
    ds_list = sorted(ppr_df['dataset'].unique())
    n_ds = len(ds_list)

    fig, axes = plt.subplots(1, n_ds, figsize=(5.5 * n_ds, 4), squeeze=False)
    for i, ds in enumerate(ds_list):
        ax = axes[0, i]
        for enc in sorted(ppr_df['encoder'].unique()):
            sub = ppr_df[(ppr_df['dataset'] == ds) & (ppr_df['encoder'] == enc)]
            sub = sub.sort_values('top_k')
            ax.plot(sub['top_k'], sub['mrr'], marker='o', markersize=5,
                    color=ENCODER_COLORS.get(enc, '#999'), label=enc)
        ax.set_xlabel('top_k'); ax.set_ylabel('MRR')
        ax.set_title(ds, fontsize=12, fontweight='bold')
        ax.legend(fontsize=9)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    fig.suptitle('Static PPR — MRR vs top_k',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No Static PPR results found.')

## 5. Static k-hop: MRR vs `k`

In [ ]:
khop_df = df[df['method'] == 'Static k-hop'].copy()
if not khop_df.empty and 'config_value' in khop_df.columns:
    khop_df['k'] = pd.to_numeric(khop_df['config_value'], errors='coerce')
    khop_df = khop_df.dropna(subset=['k'])
    ds_list = sorted(khop_df['dataset'].unique())
    n_ds = len(ds_list)

    fig, axes = plt.subplots(1, n_ds, figsize=(5.5 * n_ds, 4), squeeze=False)
    for i, ds in enumerate(ds_list):
        ax = axes[0, i]
        for enc in sorted(khop_df['encoder'].unique()):
            sub = khop_df[(khop_df['dataset'] == ds) & (khop_df['encoder'] == enc)]
            sub = sub.sort_values('k')
            ax.plot(sub['k'], sub['mrr'], marker='s', markersize=6,
                    color=ENCODER_COLORS.get(enc, '#999'), label=enc)
        ax.set_xlabel('k'); ax.set_ylabel('MRR')
        ax.set_title(ds, fontsize=12, fontweight='bold')
        ax.legend(fontsize=9)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    fig.suptitle('Static k-hop — MRR vs k',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('No k-hop results found.')

## 6. Training Time vs MRR (Pareto View)

Marker **shape** = method, **color** = encoder. Log-scale x-axis.

In [ ]:
if not df_best.empty:
    ds_list = sorted(df_best['dataset'].unique())
    n_ds = len(ds_list)

    fig, axes = plt.subplots(1, n_ds, figsize=(6 * n_ds, 5), squeeze=False)
    for i, ds in enumerate(ds_list):
        ax = axes[0, i]
        sub = df_best[df_best['dataset'] == ds]

        for _, row in sub.iterrows():
            m = METHOD_MARKERS.get(row['method'], 'x')
            c = ENCODER_COLORS.get(row['encoder'], '#999')
            ax.scatter(row['train_time'], row['mrr'],
                       marker=m, color=c, s=100, zorder=5,
                       edgecolors='black', linewidths=0.5)
            ax.annotate(f'{row["method"][:8]}\n{row["encoder"]}',
                        (row['train_time'], row['mrr']),
                        fontsize=7, alpha=0.75, ha='left',
                        xytext=(6, 2), textcoords='offset points')

        ax.set_xscale('log')
        ax.set_xlabel('Training Time (s, log scale)')
        ax.set_ylabel('MRR')
        ax.set_title(ds, fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)

    # Build combined legend
    from matplotlib.lines import Line2D
    legend_enc = [Line2D([0], [0], marker='o', color='w',
                         markerfacecolor=c, markersize=8, label=e)
                  for e, c in ENCODER_COLORS.items()]
    legend_meth = [Line2D([0], [0], marker=m, color='w',
                          markerfacecolor='gray', markersize=8, label=meth)
                   for meth, m in METHOD_MARKERS.items()]
    axes[0, -1].legend(handles=legend_enc + legend_meth,
                       loc='lower right', fontsize=8, ncol=1)

    fig.suptitle('MRR vs Training Time (best config)',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

## 7. Cross-Dataset: Relative MRR (vs Full Graph)

Each bar = MRR of method / MRR of Full Graph for the same (dataset, encoder).
Values > 1 mean the method **outperforms** the full-graph baseline.

In [ ]:
if not df_best.empty and 'Full Graph' in df_best['method'].values:
    fg = df_best[df_best['method'] == 'Full Graph'][['dataset', 'encoder', 'mrr']]
    fg = fg.rename(columns={'mrr': 'fg_mrr'})
    merged = df_best.merge(fg, on=['dataset', 'encoder'], how='left')
    merged['rel_mrr'] = merged['mrr'] / merged['fg_mrr'].clip(lower=1e-8)
    merged = merged[merged['method'] != 'Full Graph']

    ds_list = sorted(merged['dataset'].unique())
    n_ds = len(ds_list)
    fig, axes = plt.subplots(1, n_ds, figsize=(6 * n_ds, 4.5), squeeze=False,
                             sharey=True)

    for i, ds in enumerate(ds_list):
        ax = axes[0, i]
        sub = merged[merged['dataset'] == ds].sort_values('rel_mrr', ascending=True)
        labels = sub['method'] + '\n' + sub['encoder']
        colors = [ENCODER_COLORS.get(e, '#999') for e in sub['encoder']]
        ax.barh(range(len(sub)), sub['rel_mrr'], color=colors, alpha=0.85)
        ax.axvline(1.0, color='black', linestyle='--', linewidth=1, alpha=0.5)
        ax.set_yticks(range(len(sub)))
        ax.set_yticklabels(labels, fontsize=9)
        ax.set_xlabel('MRR / Full Graph MRR')
        ax.set_title(ds, fontsize=12, fontweight='bold')

    fig.suptitle('Relative MRR vs Full Graph Baseline',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('Need Full Graph results to compute relative MRR.')

## 8. Hits@k Ladder (Best Config)

In [ ]:
hits_cols = ['hits@1', 'hits@3', 'hits@10', 'hits@50']
avail_hits = [c for c in hits_cols if c in df_best.columns and df_best[c].sum() > 0]

if not df_best.empty and avail_hits:
    ds_list = sorted(df_best['dataset'].unique())
    for ds in ds_list:
        sub = df_best[df_best['dataset'] == ds].copy()
        sub['label'] = sub['encoder'] + ' / ' + sub['method']
        sub = sub.sort_values('mrr', ascending=False)

        fig, ax = plt.subplots(figsize=(max(8, 1.8 * len(sub)), 5))
        x = np.arange(len(sub))
        width = 0.8 / len(avail_hits)

        for j, hk in enumerate(avail_hits):
            ax.bar(x + j * width, sub[hk], width, label=hk)

        ax.set_xticks(x + width * (len(avail_hits) - 1) / 2)
        ax.set_xticklabels(sub['label'], rotation=35, ha='right', fontsize=9)
        ax.set_ylabel('Score')
        ax.set_title(f'{ds} — Hits@k Comparison (best config)',
                     fontsize=13, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3, axis='y')
        plt.tight_layout()
        plt.show()
else:
    print('No Hits@k data available.')

## 8e. MRR Heatmap (Encoder x Method)

In [ ]:
if not df_best.empty:
    ds_list = sorted(df_best['dataset'].unique())
    for ds in ds_list:
        sub = df_best[df_best['dataset'] == ds]
        pivot = sub.pivot_table(
            index='encoder', columns='method', values='mrr', aggfunc='max')
        if pivot.empty: continue

        fig, ax = plt.subplots(figsize=(max(6, 1.6 * pivot.shape[1]), 3.5))
        sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlGnBu',
                    cbar_kws={'label': 'MRR'}, ax=ax, linewidths=0.5)
        ax.set_title(f'{ds} — MRR Heatmap',
                     fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

## 9. Full Metric Table (All Configs)

Every experiment row including sub-configs — for audit and appendix.

In [ ]:
if not df.empty:
    metric_cols = ['mrr', 'auc', 'ap', 'hits@1', 'hits@3', 'hits@10',
                   'hits@50', 'hits@100']
    avail = [c for c in metric_cols if c in df.columns]
    show = ['dataset', 'encoder', 'method', 'config'] + avail + ['train_time']
    show = [c for c in show if c in df.columns]
    tbl = df[show].sort_values(['dataset', 'method', 'config', 'encoder'])

    fmt = {m: '{:.4f}' for m in avail}
    fmt['train_time'] = '{:.0f}s'

    styled = tbl.style.format(fmt)
    display(styled)
else:
    print('No data.')

## 10. Save Analysis Outputs

In [ ]:
ANALYSIS_DIR = 'results/benchmark-analysis'
os.makedirs(ANALYSIS_DIR, exist_ok=True)

if not df.empty:
    df.to_csv(os.path.join(ANALYSIS_DIR, 'full_table.csv'), index=False)
    print(f'Saved full_table.csv ({len(df)} rows)')

if not df_best.empty:
    df_best.to_csv(os.path.join(ANALYSIS_DIR, 'best_config_table.csv'), index=False)
    print(f'Saved best_config_table.csv ({len(df_best)} rows)')

    winners = []
    for (ds, enc), grp in df_best.groupby(['dataset', 'encoder']):
        best = grp.loc[grp['mrr'].idxmax()]
        winners.append({
            'dataset': ds, 'encoder': enc,
            'best_method': best['method'],
            'best_config': best['config'],
            'mrr': best['mrr'],
        })
    pd.DataFrame(winners).to_csv(
        os.path.join(ANALYSIS_DIR, 'winners.csv'), index=False)
    print(f'Saved winners.csv')

print(f'\nAll analysis outputs saved to {ANALYSIS_DIR}/')

## 11. Conclusion

In [ ]:
if not df_best.empty:
    print('=' * 80)
    print('BENCHMARK COMPARISON — SUMMARY')
    print('=' * 80)

    methods = sorted(df_best['method'].unique())
    ds_list = sorted(df_best['dataset'].unique())
    enc_list = sorted(df_best['encoder'].unique())

    # Win rate
    wins = defaultdict(int)
    total = 0
    for (ds, enc), grp in df_best.groupby(['dataset', 'encoder']):
        wins[grp.loc[grp['mrr'].idxmax()]['method']] += 1
        total += 1
    print(f'\nWin rate by MRR ({total} cells):')
    for m in sorted(wins, key=wins.get, reverse=True):
        print(f'  {m:20s}: {wins[m]}/{total} ({100*wins[m]/total:.0f}%)')

    # Per-dataset summary
    for ds in ds_list:
        print(f'\n--- {ds} ---')
        for enc in enc_list:
            sub = df_best[(df_best['dataset'] == ds) & (df_best['encoder'] == enc)]
            if sub.empty: continue
            best = sub.loc[sub['mrr'].idxmax()]
            print(f'  {enc:5s} | {best["method"]:16s} ({best["config"]:>12s}) | '
                  f'MRR={best["mrr"]:.4f} | Time={best["train_time"]:.0f}s')

    # Subgraph benefit
    if 'Full Graph' in df_best['method'].values:
        fg = df_best[df_best['method'] == 'Full Graph']
        others = df_best[df_best['method'] != 'Full Graph']
        if not fg.empty and not others.empty:
            fg_avg = fg['mrr'].mean()
            others_best_avg = others.groupby(['dataset', 'encoder'])['mrr'].max().mean()
            delta = others_best_avg - fg_avg
            print(f'\nAvg MRR: Full Graph = {fg_avg:.4f}, '
                  f'Best Subgraph = {others_best_avg:.4f} '
                  f'(delta = {delta:+.4f})')
            if delta > 0.001:
                print('  -> Subgraph extraction provides a measurable benefit.')
            elif delta > -0.001:
                print('  -> Subgraph and full-graph are comparable.')
            else:
                print('  -> Full graph outperforms subgraph methods on average.')

    print('\n' + '=' * 80)
else:
    print('No results to summarize.')

### Interpretation Guide

**Q1: Does subgraph extraction help?**
Check the relative MRR chart (Section 7). Bars > 1.0 mean the subgraph
method beats the full-graph baseline for that (dataset, encoder).

**Q2: Which subgraph strategy is best?**
The heatmap (Section 8e) and leaderboard (Section 1) give a direct answer.
If Learnable PPR consistently wins, the architecture search overhead is
justified.

**Q3: Is the extra training time worth it?**
The Pareto scatter (Section 6) shows cost vs benefit. Points in the
upper-left are ideal (high MRR, low time).

**Q4: How sensitive are static methods to their hyperparameter?**
PPR sensitivity (Section 4) and k-hop sensitivity (Section 5) show this.
Flat curves = robust; steep curves = careful tuning required.

### Reproducibility

All experiments were run with `SEED = 42` in `benchmark_runner.ipynb`.
For publication, re-run with 3-5 seeds and report mean +/- std.

### Limitations

- Single seed (extend for significance testing)
- No learnable k-hop baseline (not implemented in this codebase)
- Learnable PPR uses first-order DARTS approximation